In [27]:
import numpy as np
from collections import defaultdict

def gf2_gauss_elimination(A):
    """
    Приведение матрицы к ступенчатому виду над GF(2).
    Возвращает приведенную матрицу и список позиций ведущих элементов (pivots).
    """
    M = A.copy() % 2
    rows, cols = M.shape
    pivots = []
    r = 0
    
    for c in range(cols):
        if r >= rows:
            break
        # Поиск ведущей 1 в столбце c
        pivot_row = -1
        for i in range(r, rows):
            if M[i, c] == 1:
                pivot_row = i
                break
        
        if pivot_row == -1:
            continue
        
        # Перестановка строк
        M[[r, pivot_row]] = M[[pivot_row, r]]
        pivots.append(c)
        
        # Исключение 1 в столбце c для всех остальных строк
        for i in range(rows):
            if i != r and M[i, c] == 1:
                M[i] = (M[i] + M[r]) % 2
        
        r += 1
    
    return M, pivots

def get_null_space_basis(G):
    """
    Нахождение базиса нуль-пространства матрицы G над GF(2)
    методом Гаусса-Жордана.
    """
    m, n = G.shape
    
    # Приводим G к ступенчатому виду
    R, pivots = gf2_gauss_elimination(G)
    
    # Определяем свободные переменные (столбцы без ведущих элементов)
    free_cols = [c for c in range(n) if c not in pivots]
    num_free = len(free_cols)
    
    # Размерность нуль-пространства должна быть n - rank(G)
    rank = len(pivots)
    
    if num_free == 0:
        return np.zeros((0, n), dtype=int)
    
    # Строим базисные векторы для каждой свободной переменной
    basis = []
    
    for free_col in free_cols:
        vec = np.zeros(n, dtype=int)
        vec[free_col] = 1  # Устанавливаем свободную переменную в 1
        
        # Выражаем зависимые переменные через свободные
        # Из приведенной матрицы R: для каждой строки i с pivot pivots[i]
        # R[i, pivots[i]] = 1, и сумма R[i, j]*x[j] = 0
        for i, pivot_col in enumerate(pivots):
            # Если в строке i есть 1 в столбце free_col, то зависимая переменная
            # должна компенсировать это
            if R[i, free_col] == 1:
                vec[pivot_col] = 1
        
        basis.append(vec)
    
    return np.array(basis)

def generate_codewords(G):
    """Генерация всех кодовых слов по порождающей матрице."""
    k, n = G.shape
    codewords = []
    for i in range(2**k):
        u = np.array([(i >> j) & 1 for j in range(k-1, -1, -1)])
        c = np.dot(u, G) % 2
        codewords.append(c)
    return np.array(codewords)

def build_trellis_profile_from_codewords(codewords):
    """
    Построение профиля минимальной решетки по множеству кодовых слов.
    Состояние на ярусе i определяется множеством возможных суффиксов.
    """
    n = len(codewords[0])
    profile = []
    
    for i in range(n + 1):
        suffixes_map = defaultdict(list)
        for cw in codewords:
            prefix = tuple(cw[:i])
            suffix = tuple(cw[i:])
            suffixes_map[prefix].append(suffix)
        
        # Группируем префиксы с одинаковыми множествами суффиксов
        unique_future_sets = set()
        for prefix, suffixes in suffixes_map.items():
            future_set = frozenset(suffixes)
            unique_future_sets.add(future_set)
        
        profile.append(len(unique_future_sets))
        
    return profile

def build_syndrome_trellis_profile(H):
    """
    Построение профиля синдромной решетки.
    Состояния - валидные частичные синдромы.
    """
    if H.shape[0] == 0:
        return None
        
    m, n = H.shape
    profile = []
    
    h_cols = [H[:, j] for j in range(n)]
    
    # Прямое достижение (Forward)
    S = [set() for _ in range(n + 1)]
    S[0].add(tuple(np.zeros(m, dtype=int)))
    
    for i in range(n):
        for s in S[i]:
            s_vec = np.array(s)
            S[i+1].add(tuple(s_vec))  # Ветвь 0
            s_new = (s_vec + h_cols[i]) % 2  # Ветвь 1
            S[i+1].add(tuple(s_new))
            
    # Обратное достижение (Backward)
    E = [set() for _ in range(n + 1)]
    E[n].add(tuple(np.zeros(m, dtype=int)))
    
    for i in range(n - 1, -1, -1):
        for s in E[i+1]:
            s_vec = np.array(s)
            E[i].add(tuple(s_vec))  # Ветвь 0
            s_prev = (s_vec + h_cols[i]) % 2  # Ветвь 1
            E[i].add(tuple(s_prev))
            
    # Валидные узлы
    for i in range(n + 1):
        valid_states = S[i].intersection(E[i])
        profile.append(len(valid_states))
        
    return profile

In [28]:
G = np.array([
    [1, 0, 1, 1, 0, 1],
    [1, 0, 1, 0, 1, 0],
    [1, 1, 0, 1, 0, 0]
])

k, n = G.shape
print(f"Параметры кода: n={n}, k={k}")

Параметры кода: n=6, k=3


# 1. Генерация кодовых слов

In [29]:
codewords = generate_codewords(G)
print(f"Количество кодовых слов: {len(codewords)}")

Количество кодовых слов: 8


# 2. Построение решетки по G

In [30]:
profile_G = build_trellis_profile_from_codewords(codewords)
print(f"Профиль решетки по G: {profile_G}")

Профиль решетки по G: [1, 2, 4, 4, 4, 2, 1]


# 3. Вычисление проверочной матрицы H

In [31]:
H = get_null_space_basis(G)
print(f"\nРазмер проверочной матрицы H: {H.shape}")
print("Матрица H:")
print(H)


Размер проверочной матрицы H: (3, 6)
Матрица H:
[[1 1 1 0 0 0]
 [1 0 0 1 1 0]
 [0 1 0 1 0 1]]


# 4. Валидация $G * H^T = 0$


In [32]:
validation = np.dot(G, H.T) % 2
print(f"\nПроверка ортогональности (G * H^T mod 2):\n{validation}")
if np.all(validation == 0):
    print("Ортогональность подтверждена.")
else:
    print("ОШИБКА: Ортогональность НЕ подтверждена!")


Проверка ортогональности (G * H^T mod 2):
[[0 0 0]
 [0 0 0]
 [0 0 0]]
Ортогональность подтверждена.


# 5. Построение синдромной решетки

In [33]:
if H.shape[0] > 0:
    profile_H = build_syndrome_trellis_profile(H)
    print(f"\nПрофиль синдромной решетки: {profile_H}")
    
    # 6. Сравнение
    print("\n=== Верификация ===")
    if profile_G == profile_H:
        print("Результат: Профили решеток совпадают.")
        print("Утверждение задания подтверждено.")
    else:
        print("Результат: Профили решеток НЕ совпадают.")
else:
    print("Ошибка: Матрица H пуста, невозможно построить синдромную решетку.")


Профиль синдромной решетки: [1, 2, 4, 4, 4, 2, 1]

=== Верификация ===
Результат: Профили решеток совпадают.
Утверждение задания подтверждено.
